In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries loaded successfully")

In [ ]:
df = pd.read_csv('../data/bank_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print("=== Dataset Info ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Statistics ===")
df.describe()


In [ ]:
cols_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Columns after dropping irrelevant ones: {list(df.columns)}")

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

# Fill numeric missing values with median (robust to outliers)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f"  Filled missing values in '{col}' with median")

print("\nMissing values after handling:")
print(df.isnull().sum())

In [ ]:
print("Categorical columns:", df.select_dtypes(include='object').columns.tolist())

# One-hot encode Geography (France, Spain, Germany -> 3 binary columns)
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

# Binary encode Gender (Male=1, Female=0)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

print(f"\nDataset shape after encoding: {df.shape}")
print(f"New columns: {list(df.columns)}")

In [ ]:
churn_counts = df['Exited'].value_counts()
churn_pct    = df['Exited'].value_counts(normalize=True) * 100

print("=== Class Distribution ===")
print(f"  Not Churned (0): {churn_counts[0]}  ({churn_pct[0]:.1f}%)")
print(f"  Churned     (1): {churn_counts[1]}  ({churn_pct[1]:.1f}%)")

# Visualise
plt.figure(figsize=(5, 3))
churn_counts.plot(kind='bar', color=['#378ADD', '#D85A30'], edgecolor='none')
plt.title('Class Distribution — Churn vs Not Churned')
plt.xlabel('Exited (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../report/class_distribution.png', dpi=150)
plt.show()
print("Plot saved to report/")

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited']

print(f"Features (X) shape: {X.shape}")
print(f"Target  (y) shape:  {y.shape}")
print(f"\nFeature columns:\n{list(X.columns)}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,       # fixed seed = reproducible results
    stratify=y             # preserve class imbalance ratio
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nChurn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.1f}%")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert back to DataFrames so column names are preserved
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

print("Scaling complete.")
print(f"\nSample means after scaling (should be ~0):")
print(X_train_scaled.mean().round(6).head(5))

In [ ]:
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv',   index=False)
y_train.to_csv('../data/y_train.csv',         index=False)
y_test.to_csv('../data/y_test.csv',           index=False)

print("Processed data saved to data/ folder:")
print("  X_train.csv, X_test.csv, y_train.csv, y_test.csv")


In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr()
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='Blues',
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../report/correlation_heatmap.png', dpi=150)
plt.show()
print("Heatmap saved to report/")

print("\n=== Preprocessing Complete ===")
print("All other members can now load X_train.csv / X_test.csv")
